In [1]:
%matplotlib notebook
from rfsoc_rfdc.rfsoc_overlay import RFSoCOverlay
from rfsoc_rfdc.overlay_task import OverlayTask
from rfsoc_rfdc.overlay_task import BlinkLedTask

from rfsoc_rfdc.transmitter.single_ch_tx_task import SingleChTxTask
from rfsoc_rfdc.receiver.single_ch_rx_task import SingleChRxTask

from rfsoc_rfdc.transmitter.multi_ch_tx_mimo_task import MultiChTxMIMOTask
from rfsoc_rfdc.receiver.multi_ch_rx_mimo_task import MultiChRxMIMOTask

from rfsoc_rfdc.rfdc_task import RfdcTask 
from rfsoc_rfdc.cmac_task import CmacTask

from rfsoc_rfdc.rfdc_config import ZCU216_CONFIG

import sys
import os
import time

In [2]:
from rfsoc_rfdc.dsp.ofdm import OFDM
from rfsoc_rfdc.dsp.mimo_detection import MIMODetection

In [3]:
ol = RFSoCOverlay(path_to_bitstream="./rfsoc_rfdc/bitstream/rfsoc_rfdc_v45_rt_2ch_mts.bit")
NEW_CONFIG = {
    "RefClockForPLL": 300.0,
    "DACSampleRate": 2400.0,
    "DACInterpolationRate": 4,
    "DACNCO": 800,
    "ADCSampleRate": 2400.0,
    "ADCInterpolationRate": 4,
    "ADCNCO": -800                                           
}
ZCU216_CONFIG.update(NEW_CONFIG)

MTS_block/MTS_clk_wiz
adc_datapath/data_mover_ctrl
adc_datapath/fifo_count
axi_gpio_led
axi_intp
dac_datapath/data_mover_ctrl
dac_datapath/fifo_count
rfdc/usp_rf_data_converter
zynq_ultra_ps_e


In [4]:
true_samp_rate = ZCU216_CONFIG['DACSampleRate'] / ZCU216_CONFIG['DACInterpolationRate'] * 1e6

In [5]:
rfdc_t = RfdcTask(ol, mts_enable=True, debug_mode=True, board="ZCU216")

for task in [rfdc_t]:
    task.start()
    task.join()

2025-05-31 00:24:31,354 - root - WARNING - DAC NCO shall fall between -1/2*Fs and 1/2*Fs
2025-05-31 00:24:31,356 - root - WARNING - ADC NCO shall fall between 0 and 1/2*Fs
2025-05-31 00:24:31,572 - root - INFO - DAC tile 0 DAC block 0 is enabled!
2025-05-31 00:24:31,612 - root - INFO - DAC tile 0 DAC block 1 is enabled!
2025-05-31 00:24:31,655 - root - INFO - DAC tile 0 DAC block 2 is enabled!
2025-05-31 00:24:31,695 - root - INFO - DAC tile 0 DAC block 3 is enabled!
2025-05-31 00:24:31,734 - root - INFO - DAC tile 1 DAC block 0 is enabled!
2025-05-31 00:24:31,774 - root - INFO - DAC tile 1 DAC block 1 is enabled!
2025-05-31 00:24:31,814 - root - INFO - DAC tile 1 DAC block 2 is enabled!
2025-05-31 00:24:31,855 - root - INFO - DAC tile 1 DAC block 3 is enabled!
2025-05-31 00:24:31,894 - root - INFO - DAC tile 2 DAC block 0 is enabled!
2025-05-31 00:24:31,934 - root - INFO - DAC tile 2 DAC block 1 is enabled!
2025-05-31 00:24:31,972 - root - INFO - DAC tile 2 DAC block 2 is enabled!
202

In [6]:
for mcs in range(8):
    
    for atten in range(0, 28, 3):
        
        ZCU216_CONFIG['OFDM_ATTEN_DB'] = atten
        ZCU216_CONFIG['DETECTION_SCHEME'] = MIMODetection(sample_rate=true_samp_rate, tx_num=2, rx_num=2, MCS=mcs)
        ZCU216_CONFIG['CONFIG_NAME'] = f"Atten{atten}"

        print(f"MCS={mcs}, OFDM_ATTEN_DB={atten}")

        led_t = BlinkLedTask(ol)
        tx_t = MultiChTxMIMOTask(ol, mode="iq2real", channel_count=2, dp_vect_dim=2)
        rx_t = MultiChRxMIMOTask(ol, mode="real2iq", channel_count=2, dp_vect_dim=2)

        parallel_task = [led_t, tx_t, rx_t]
        for task in parallel_task:
            task.start()

        time.sleep(20)

        for task in parallel_task:
            task.stop()

MCS=0, OFDM_ATTEN_DB=0


FigureWidget({
    'data': [],
    'layout': {'template': '...',
               'title': {'text': 'TimeDomainA…

FigureWidget({
    'data': [],
    'layout': {'template': '...',
               'title': {'text': 'TimeDomainA…

2025-05-31 00:24:35,389 - root - INFO - Tx waveform waveTx_2_0.mat loaded successfully.
2025-05-31 00:24:35,546 - root - INFO - Tx data preparation done.


2025-05-31 00:24:38,998 - root - INFO - Rx samples captured.
/mnt/sata_ssd_128g/jupyter_notebooks/SPEAR/rfsoc_rfdc/dsp/mimo_detection.py:144: RuntimeWarning:

invalid value encountered in double_scalars

2025-05-31 00:24:41,587 - root - WARNING - Rx detection failed.
2025-05-31 00:24:42,277 - root - INFO - Rx samples captured.
2025-05-31 00:24:47,391 - root - INFO - Rx #CH0 SNR: 33.048, CFO: 0.000
2025-05-31 00:24:47,394 - root - INFO - Rx #CH1 SNR: 31.658, CFO: 0.000
2025-05-31 00:24:48,082 - root - INFO - Rx samples captured.
2025-05-31 00:24:52,844 - root - INFO - Rx #CH0 SNR: 34.583, CFO: 0.000
2025-05-31 00:24:52,847 - root - INFO - Rx #CH1 SNR: 33.144, CFO: 0.000
2025-05-31 00:24:53,533 - root - INFO - Rx samples captured.
2025-05-31 00:24:58,312 - root - INFO - Rx #CH0 SNR: 32.859, CFO: 0.000
2025-05-31 00:24:58,314 - root - INFO - Rx #CH1 SNR: 32.710, CFO: 0.000


...

2025-05-31 00:55:39,927 - root - INFO - Tx waveform waveTx_2_7.mat loaded successfully.
2025-05-31 00:55:40,077 - root - INFO - Tx data preparation done.
2025-05-31 00:55:43,408 - root - INFO - Rx samples captured.
2025-05-31 00:55:47,823 - root - INFO - Rx #CH0 SNR: 15.412, CFO: 0.000
2025-05-31 00:55:47,825 - root - INFO - Rx #CH1 SNR: 14.039, CFO: 0.000
2025-05-31 00:55:48,498 - root - INFO - Rx samples captured.
2025-05-31 00:55:52,858 - root - INFO - Rx #CH0 SNR: 13.908, CFO: 0.000
2025-05-31 00:55:52,861 - root - INFO - Rx #CH1 SNR: 12.772, CFO: 0.000
2025-05-31 00:55:53,540 - root - INFO - Rx samples captured.
2025-05-31 00:55:57,922 - root - INFO - Rx #CH0 SNR: 13.984, CFO: 0.000
2025-05-31 00:55:57,924 - root - INFO - Rx #CH1 SNR: 12.565, CFO: 0.000
2025-05-31 00:55:58,611 - root - INFO - Rx samples captured.
2025-05-31 00:56:03,007 - root - INFO - Rx #CH0 SNR: 14.078, CFO: 0.000
2025-05-31 00:56:03,010 - root - INFO - Rx #CH1 SNR: 12.562, CFO: 0.000


MCS=7, OFDM_ATTEN_DB=27


FigureWidget({
    'data': [],
    'layout': {'template': '...',
               'title': {'text': 'TimeDomainA…

FigureWidget({
    'data': [],
    'layout': {'template': '...',
               'title': {'text': 'TimeDomainA…

2025-05-31 00:56:03,567 - root - INFO - Tx waveform waveTx_2_7.mat loaded successfully.
2025-05-31 00:56:03,714 - root - INFO - Tx data preparation done.
2025-05-31 00:56:07,079 - root - INFO - Rx samples captured.
2025-05-31 00:56:09,404 - root - WARNING - Rx detection failed.
2025-05-31 00:56:10,104 - root - INFO - Rx samples captured.
2025-05-31 00:56:14,451 - root - INFO - Rx #CH0 SNR: 8.791, CFO: 0.000
2025-05-31 00:56:14,454 - root - INFO - Rx #CH1 SNR: 8.746, CFO: 0.000
2025-05-31 00:56:15,118 - root - INFO - Rx samples captured.
2025-05-31 00:56:19,498 - root - INFO - Rx #CH0 SNR: 11.103, CFO: 0.000
2025-05-31 00:56:19,501 - root - INFO - Rx #CH1 SNR: 9.699, CFO: 0.000
2025-05-31 00:56:20,178 - root - INFO - Rx samples captured.
2025-05-31 00:56:24,564 - root - INFO - Rx #CH0 SNR: 10.870, CFO: 0.000
2025-05-31 00:56:24,568 - root - INFO - Rx #CH1 SNR: 9.796, CFO: 0.000
2025-05-31 00:56:25,280 - root - INFO - Rx samples captured.
2025-05-31 00:56:25,943 - root - WARNING - Captur